# Fase III — Entrenamiento GNN-GIN sobre Tox21

Este notebook entrena la **GNN-GIN** (`GINToxicity`) sobre los grafos moleculares pre-procesados
de Tox21 y evalúa el rendimiento en el conjunto de test (scaffold split).

**Requisitos previos:**
- `make prepare-graphs` o `python scripts/fase1/prepare_tox21_graphs.py`
- Baselines entrenados (Fase II) para comparación opcional

**Objetivo:** AUC-ROC medio en test > 0.82, superando RF (~0.77), MLP (~0.79) y SMILES2vec (~0.81).

| Bloque | Descripción |
|---|---|
| 1 | Proyección inicial: Linear(45 → d) + BatchNorm |
| 2 | Message passing GINEConv (L capas) + residuales |
| 3 | Readout global: mean + max pooling |
| 4 | Clasificador multitarea: 12 logits (una por diana Tox21) |

## 0. Configuración

In [1]:
import random
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from IPython.display import display
from torch_geometric.loader import DataLoader

from scripts.train_gin import (
    build_loaders,
    compute_pos_weight,
    load_config,
    resolve_device,
    save_results,
    set_seed,
)
from src.data.dataset import N_TASKS, TASK_NAMES, ToxicityDataset
from src.evaluation.cross_validation import evaluate_multitask_auc
from src.models.gin import GINToxicity
from src.training.loss import MaskedBCELoss
from src.training.trainer import evaluate, train_epoch

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 5)})

CONFIG_PATH = ROOT / "config" / "config.yaml"
DATA_DIR = ROOT / "data" / "processed"
OUT_DIR = ROOT / "outputs" / "results"
FIG_DIR = ROOT / "outputs" / "gin"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
USE_WANDB = False  # cambiar a True si tienes wandb configurado

cfg = load_config(CONFIG_PATH)
device = resolve_device("auto", require_gpu=False)
set_seed(SEED, device)

print(f"Raíz: {ROOT}")
print(f"Dispositivo: {device}")
print(f"Config: hidden_dim={cfg['model']['hidden_dim']}, n_layers={cfg['model']['n_layers']}")

c:\Users\mateo\Desktop\PROYECTOS\JIC2026\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'scripts.train_gin'

## 1. Carga de grafos moleculares

In [ ]:
for split in ("train", "val", "test"):
    p = DATA_DIR / f"graphs_{split}.pt"
    if not p.is_file():
        raise FileNotFoundError(
            f"No existe {p}. Ejecuta: make prepare-graphs"
        )

batch_size = int(cfg["training"]["batch_size"])
train_loader, val_loader, test_loader, train_ds, n_tr, n_va, n_te = build_loaders(
    DATA_DIR, batch_size, device, num_workers=0
)

print(f"Train: {n_tr:,} | Val: {n_va:,} | Test: {n_te:,}")
print(f"Batch size: {batch_size} | Tareas: {N_TASKS}")

sample = train_ds[0]
print(f"Ejemplo — nodos: {sample.x.shape[0]}, aristas: {sample.edge_index.shape[1]}")

## 2. Modelo y compensación de desbalance (`pos_weight`)

Tox21 tiene muy pocos positivos por tarea (2–5% en NR-AR). Sin `pos_weight`, el modelo
aprende a predecir "no tóxico" para todo.

In [ ]:
model_cfg = cfg["model"]
model = GINToxicity(
    node_feat_dim=int(model_cfg["node_feat_dim"]),
    edge_feat_dim=int(model_cfg["edge_feat_dim"]),
    hidden_dim=int(model_cfg["hidden_dim"]),
    n_layers=int(model_cfg["n_layers"]),
    n_tasks=int(model_cfg["n_tasks"]),
    dropout=float(model_cfg["dropout"]),
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {n_params:,}")

pos_weight = None
if cfg["training"].get("use_pos_weight", False):
    pos_weight = compute_pos_weight(train_ds)
    pw_df = pd.DataFrame({"tarea": TASK_NAMES, "pos_weight": pos_weight.numpy()})
    display(pw_df)
else:
    print("pos_weight desactivado en config.yaml")

## 3. Entrenamiento con early stopping

El mejor checkpoint (por `val_auc`) se guarda en `outputs/models/best_gin_model.pt`.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=float(cfg["training"]["lr"]))
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt,
    mode="max",
    factor=float(cfg["scheduler"]["factor"]),
    patience=int(cfg["scheduler"]["patience"]),
)
pw = pos_weight.to(device) if pos_weight is not None else None
loss_fn = MaskedBCELoss(pos_weight=pw)

save_path = Path(cfg["training"]["model_save_path"])
save_path.parent.mkdir(parents=True, exist_ok=True)

max_epochs = int(cfg["training"]["max_epochs"])
patience = int(cfg["training"]["early_stopping_patience"])
grad_clip = float(cfg["training"]["grad_clip_norm"])

if USE_WANDB:
    import wandb
    wandb.init(project=cfg["wandb"]["project"], entity=cfg["wandb"].get("entity"), config=cfg)

history = {"epoch": [], "train_loss": [], "val_auc": [], "lr": []}
best_val = 0.0
bad = 0

print("=== Entrenamiento GIN ===")
t0 = time.time()

for epoch in range(max_epochs):
    tl = train_epoch(model, train_loader, opt, loss_fn, device, grad_clip)
    _, val_auc = evaluate(model, val_loader, device, TASK_NAMES)

    if not np.isfinite(val_auc):
        bad += 1
        print(f"  Época {epoch + 1}/{max_epochs} — loss: {tl:.4f} val_auc: nan")
        if bad >= patience:
            print(f"Early stopping en época {epoch}. Mejor val_AUC: {best_val:.4f}")
            break
        continue

    sched.step(val_auc)
    history["epoch"].append(epoch + 1)
    history["train_loss"].append(tl)
    history["val_auc"].append(val_auc)
    history["lr"].append(opt.param_groups[0]["lr"])

    if USE_WANDB:
        wandb.log({"epoch": epoch, "train_loss": tl, "val_auc": val_auc, "lr": history["lr"][-1]})

    improved = val_auc > best_val
    if improved:
        best_val = val_auc
        bad = 0
        torch.save(model.state_dict(), save_path)
    else:
        bad += 1

    marker = " *" if improved else ""
    print(f"  Época {epoch + 1}/{max_epochs} — loss: {tl:.4f} val_auc: {val_auc:.4f}{marker}")

    if bad >= patience:
        print(f"Early stopping en época {epoch}. Mejor val_AUC: {best_val:.4f}")
        break

elapsed = time.time() - t0
print(f"\nTiempo total: {elapsed / 60:.1f} min | Mejor val_AUC: {best_val:.4f}")

if USE_WANDB:
    wandb.finish()

hist_df = pd.DataFrame(history)
hist_df.tail()

## 4. Evaluación en test

In [ ]:
if save_path.is_file():
    try:
        state = torch.load(save_path, map_location=device, weights_only=True)
    except TypeError:
        state = torch.load(save_path, map_location=device)
    model.load_state_dict(state)
else:
    print(f"[AVISO] No se encontró {save_path}; evaluando último estado.")

auc_test, mean_test = evaluate(model, test_loader, device, TASK_NAMES)

print(f"Val AUC (mejor):  {best_val:.4f}")
print(f"Test AUC (media): {mean_test:.4f}\n")
for task, auc in auc_test.items():
    print(f"  {task:16s} {auc:.4f}")

gin_results_path = OUT_DIR / "gin_results.csv"
save_results(auc_test, mean_test, best_val, gin_results_path)
print(f"\nResultados guardados en {gin_results_path}")

## 5. Visualización y comparación con baselines

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if len(hist_df) > 0:
    axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="train_loss")
    axes[0].set_xlabel("Época")
    axes[0].set_ylabel("Pérdida")
    axes[0].set_title("Curva de entrenamiento")
    axes[0].legend()

    axes[1].plot(hist_df["epoch"], hist_df["val_auc"], color="C1", label="val_auc")
    axes[1].axhline(best_val, color="gray", ls="--", label=f"mejor = {best_val:.3f}")
    axes[1].set_xlabel("Época")
    axes[1].set_ylabel("AUC-ROC")
    axes[1].set_title("Validación")
    axes[1].legend()
else:
    axes[0].text(0.5, 0.5, "Sin historial", ha="center", va="center")
    axes[1].text(0.5, 0.5, "Sin historial", ha="center", va="center")

plt.tight_layout()
fig.savefig(FIG_DIR / "training_curves.png", bbox_inches="tight")
plt.show()

auc_df = pd.DataFrame({"tarea": list(auc_test.keys()), "AUC_GIN": list(auc_test.values())})

baseline_path = OUT_DIR / "baseline_results.csv"
if baseline_path.is_file():
    bl = pd.read_csv(baseline_path)
    bl_row = bl.loc[bl["model"] == "RandomForest"]
    if not bl_row.empty:
        rf_aucs = [bl_row[task].iloc[0] for task in TASK_NAMES if task in bl.columns]
        if len(rf_aucs) == len(TASK_NAMES):
            auc_df["AUC_RF"] = rf_aucs

auc_melt = auc_df.melt(id_vars="tarea", var_name="modelo", value_name="AUC")
plt.figure(figsize=(12, 5))
sns.barplot(data=auc_melt, x="tarea", y="AUC", hue="modelo")
plt.xticks(rotation=45, ha="right")
plt.axhline(0.82, color="red", ls=":", label="objetivo 0.82")
plt.title(f"AUC-ROC por tarea — GIN test mean = {mean_test:.3f}")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "auc_per_task.png", bbox_inches="tight")
plt.show()

if mean_test >= 0.82:
    print("Objetivo alcanzado: AUC test >= 0.82")
elif mean_test >= 0.77:
    print("Supera RF baseline; revisar hiperparámetros para alcanzar 0.82")
else:
    print("[ALERTA] AUC por debajo del RF — revisar datos o configuración")